In [38]:
import pandas as pd
import time
import joblib
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# 1. Load test data

In [7]:
test_data = pd.read_csv("../data/processed/test_data.csv")

In [8]:
test_data.shape


(723, 3)

In [93]:
test_data.head()

,original_index,text,sentiment
0,2636,"The mill will have capacity to produce 500,000...",neutral
1,547,HELSINKI ( AFX ) - Nokian Tyres reported a fou...,positive
2,4685,The company confirmed its estimate for lower r...,negative
3,3071,"Prior to the transaction , whose financial ter...",neutral
4,2988,L+Ænnen Tehtaat 's Food Division was reorganis...,neutral


In [9]:
test_data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.591978
positive    0.282158
negative    0.125864
Name: proportion, dtype: float64

In [11]:
texts = test_data['text'].tolist()
true_labels = test_data['sentiment'].tolist()

# 2. Load pre-trained BERT

In [12]:
# let's load the FinBERT using the pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert")
# this downloads model weights, tokenizer files, configuration

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
# Let's how the model is doing with a single sentence
pipe("The company reported flunked profits this quarter.")

[{'label': 'negative', 'score': 0.9743381142616272}]

In [10]:
# let's test on three different samples
pipe([
    "The company announced major losses and declining sales.",
    "The company achieved record revenue growth and increased profits.",
    "The company released its quarterly financial report."
])

[{'label': 'negative', 'score': 0.9708959460258484},
 {'label': 'positive', 'score': 0.9566091299057007},
 {'label': 'neutral', 'score': 0.8721367120742798}]

# 3. Run inference

In [27]:
# now it's time to test our test_data
model_pred = pipe(texts,batch_size=32)
model_pred

[{'label': 'neutral', 'score': 0.8416659235954285},
 {'label': 'positive', 'score': 0.9535892009735107},
 {'label': 'negative', 'score': 0.9721561670303345},
 {'label': 'neutral', 'score': 0.9525972008705139},
 {'label': 'neutral', 'score': 0.9383479952812195},
 {'label': 'positive', 'score': 0.9502188563346863},
 {'label': 'positive', 'score': 0.7605570554733276},
 {'label': 'neutral', 'score': 0.856540858745575},
 {'label': 'neutral', 'score': 0.7980159521102905},
 {'label': 'negative', 'score': 0.9108859300613403},
 {'label': 'neutral', 'score': 0.8927277326583862},
 {'label': 'neutral', 'score': 0.7262792587280273},
 {'label': 'positive', 'score': 0.7854838967323303},
 {'label': 'neutral', 'score': 0.8199658393859863},
 {'label': 'neutral', 'score': 0.9494116306304932},
 {'label': 'positive', 'score': 0.9385881423950195},
 {'label': 'neutral', 'score': 0.9512954354286194},
 {'label': 'negative', 'score': 0.5936258435249329},
 {'label': 'neutral', 'score': 0.9417350888252258},
 {'la

We can manually calculate the prediction accuracy by comparing the predicted labels with the true labels. However, this metric alone is not sufficient for evaluating a classifier, so we will later use `scikit-learn`'s evaluation metrics.

```python
correct_pred = []

for i, obj in enumerate(model_pred):
    if obj["label"] == true_labels[i]:
        correct_pred.append(True)

tot_correct_pred_perc = (len(correct_pred) / len(true_labels)) * 100
print(f"Accuracy: {tot_correct_pred_perc:.2f}%")
```

While this approach computes the percentage of correct predictions, it does not provide additional metrics such as precision, recall, F1-score, or a confusion matrix. To obtain a more comprehensive evaluation, we will import `scikit-learn` and use its built-in metrics.

# 4. Extract Prections

In [95]:
# First let's put the predicted labels into a list
predicted_labels = [ item["label"] for item in model_pred ]
predicted_scores = [ item["score"] for item in model_pred ]

# 5. Evaluate

In [25]:
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy: {accuracy}")
macro_f1 = f1_score(true_labels, predicted_labels, average="macro")
print(f"Macro_F1: {macro_f1}")
class_report = classification_report(true_labels, predicted_labels)
print(f"Classification report: {class_report}")
conf_matrix = confusion_matrix(true_labels, predicted_labels)
print(f"Confusion Matrix: {conf_matrix}")

Accuracy: 0.8810511756569848
Macro_F1: 0.8723083712664647
Classification report:               precision    recall  f1-score   support

    negative       0.76      0.98      0.86        91
     neutral       0.95      0.85      0.90       428
    positive       0.82      0.91      0.87       204

    accuracy                           0.88       723
   macro avg       0.85      0.91      0.87       723
weighted avg       0.89      0.88      0.88       723

Confusion Matrix: [[ 89   2   0]
 [ 26 362  40]
 [  2  16 186]]


# 6. Latency benchmark

In [30]:
# let's calculate the average latency per sample
start = time.time()
result = pipe(texts, batch_size=32)
end = time.time()
latency = end - start
avg_latency = latency / len(texts)
print(f"Total latency : {latency:.4f}s")
print(f"Avg latency per each text sample:{avg_latency:.4f}s")

Total latency : 1.9772s
Avg latency per each text sample:0.0027s


In [73]:
# The helper function to calculate the latencies
def calc_avg_latency(model_name,text):
    if model_name != "finbert":
        fn = model.predict
        text = [text]
    else:
        fn = pipe
    # let's store the latency for each call
    latencies = []
    for _ in range(10):
        start = time.time()
        response = fn(text)
        end = time.time()
        latencies.append(end - start)
    avg_latency = sum(latencies) / len(latencies)
    return avg_latency

In [79]:
# But our API might get single request, so let's measure the latency over 10 times to get clear understanding
# because, testing one time is not ideal because, it includes the model warm-up, mem allocation, cache init, etc
# warm-up
pipe("The company reported profits")
avg_latency_finbert = calc_avg_latency("finbert", "Meta shares nosedived by 35% today.")
print(f"Avg latency per request: {avg_latency_finbert}s")

Avg latency per request: 0.023839688301086424s


In [80]:
# Now let's check the latency for the earlier v1 model - TF-IDF + Linear SVM
# let's load the SVM model
model = joblib.load("../models/tfidf_linear_svm.joblib")
# warm-up 
model.predict(["The company reported profits"])
avg_latency_svm = calc_avg_latency(model, "Meta shares nosedived by 35% today.")
print(f"Avg latency per request: {avg_latency_svm}s")

Avg latency per request: 0.0013433456420898437s


# 7. Compare & Save report

In [84]:
# Now, let's make a comparision table
# firstly, we'll create an object later convert that to a df
comparison = {
    "Model" : [
        "TF-IDF + Linear SVC",
        "FinBERT"
    ],
    "Accuracy" : [
        0.7441,
        0.8811
    ],
    "Macro F1":[
        0.7077,
        0.8723
    ],
    "Latency(ms)":[
        1.08,
        23.5
    ]
}

In [88]:
comparison_df = pd.DataFrame(comparison)
# let's save this report
comparison_df.to_csv("../reports/model_comparison_v1_vs_v2.csv",index=False)